# FloodLens — Phase 1: Foundations (v3, fixed folder pairing)
### FloodNet dataset, OpenCV basics, PyTorch pipeline, pretrained U-Net, IoU

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

**What changed from v2:** FloodNet-challenge has multiple split folders
(train/val/test), each containing an images folder and a labels folder. v2 guessed
the first image folder and first mask folder it found by name, which could pair a
train images folder with a val/test masks folder — different splits, zero overlapping
IDs, 0 matched pairs. v3 fixes this by measuring actual ID overlap across every
image/mask folder combination and picking the pair with the most matches.


## 0. Setup — install dependencies

In [ ]:
!pip install -q kagglehub segmentation-models-pytorch albumentations opencv-python-headless
print("Dependencies installed.")


In [ ]:
import os
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import kagglehub
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: No GPU detected. Check Runtime > Change runtime type > GPU.")


## 1. Download FloodNet from Kaggle

In [ ]:
path = kagglehub.dataset_download("aletbm/aerial-imagery-dataset-floodnet-challenge")
print("Path to dataset files:", path)


## 2. Inspect the full folder structure\nLook for MULTIPLE img/label folders (train/val/test) — this is expected for FloodNet.

In [ ]:
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if files:
        print(f"{indent}  {len(files)} files, e.g. {files[0]}")


## 3. Find the correct image/mask pair by ID overlap
Rather than trusting folder names, this collects every candidate image folder and
every candidate mask folder, extracts numeric IDs from filenames in each, and picks
the (image_dir, mask_dir) pair with the highest number of overlapping IDs. This is
robust to however many train/val/test folders the dataset actually has.


In [ ]:
def extract_id(filename):
    match = re.match(r"(\d+)", filename)
    return match.group(1) if match else None

candidate_img_dirs = []
candidate_mask_dirs = []

for root, dirs, files in os.walk(path):
    name = os.path.basename(root).lower()
    if not files:
        continue
    if any(k in name for k in ["label", "mask", "gt", "annotation"]):
        candidate_mask_dirs.append(root)
    elif any(k in name for k in ["img", "image"]):
        candidate_img_dirs.append(root)

print(f"Found {len(candidate_img_dirs)} candidate image folder(s):")
for d in candidate_img_dirs:
    print(" ", d)
print(f"Found {len(candidate_mask_dirs)} candidate mask folder(s):")
for d in candidate_mask_dirs:
    print(" ", d)

best_pair = None
best_overlap = -1
results = []

for img_d in candidate_img_dirs:
    img_ids = set(filter(None, (extract_id(f) for f in os.listdir(img_d))))
    for mask_d in candidate_mask_dirs:
        mask_ids = set(filter(None, (extract_id(f) for f in os.listdir(mask_d))))
        overlap = len(img_ids & mask_ids)
        results.append((overlap, img_d, mask_d))
        if overlap > best_overlap:
            best_overlap = overlap
            best_pair = (img_d, mask_d)

print("\nOverlap counts for every combination (sanity check):")
for overlap, img_d, mask_d in sorted(results, reverse=True):
    print(f"  {overlap:>5} overlapping IDs  |  {os.path.basename(img_d)}  <->  {os.path.basename(mask_d)}")

image_dir, mask_dir = best_pair
print(f"\nSelected pair with {best_overlap} overlapping IDs:")
print("  image_dir:", image_dir)
print("  mask_dir:", mask_dir)

assert best_overlap > 0, "No image/mask folder pair has any overlapping IDs — inspect the folder structure above manually."


In [ ]:
# MANUAL OVERRIDE — only edit if the selected pair above still looks wrong
# image_dir = "/path/to/correct/train-org-img"
# mask_dir  = "/path/to/correct/train-label-img"
print("Using:")
print("  image_dir =", image_dir)
print("  mask_dir  =", mask_dir)


## 4. Confirm filename pattern on the selected pair

In [ ]:
image_files_preview = sorted(os.listdir(image_dir))
mask_files_preview = sorted(os.listdir(mask_dir))

print("First 5 image filenames:", image_files_preview[:5])
print("First 5 mask filenames:", mask_files_preview[:5])


## 5. OpenCV basics
Reading images, BGR vs RGB, resizing, and RGB vs HSV.


In [ ]:
sample_name = image_files_preview[0]
sample_path = os.path.join(image_dir, sample_name)

img_bgr = cv2.imread(sample_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (256, 256))
img_hsv = cv2.cvtColor(img_resized, cv2.COLOR_RGB2HSV)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_bgr)
axes[0].set_title("Loaded as BGR (wrong colors)")
axes[1].imshow(img_resized)
axes[1].set_title("Corrected to RGB, resized 256x256")
axes[2].imshow(img_hsv[:, :, 2], cmap='gray')
axes[2].set_title("HSV Value channel (brightness only)")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("Image shape:", img_resized.shape)


## 6. PyTorch Dataset and DataLoader
Single definition, ID-based matching, built on the correctly-paired folders from
Section 3.


In [ ]:
IMG_SIZE = 256
SUBSET_SIZE = 300   # set to None to use the full dataset once you're ready to scale up
BATCH_SIZE = 8

class FloodDataset(Dataset):
    def __init__(self, image_dir, mask_dir, subset_size=None, img_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size

        image_files = sorted(os.listdir(image_dir))
        mask_files = sorted(os.listdir(mask_dir))

        mask_by_id = {}
        for m in mask_files:
            mid = extract_id(m)
            if mid is not None:
                mask_by_id[mid] = m

        self.pairs = []
        unmatched = []
        for f in image_files:
            fid = extract_id(f)
            if fid is not None and fid in mask_by_id:
                self.pairs.append((f, mask_by_id[fid]))
            else:
                unmatched.append(f)

        if subset_size is not None:
            self.pairs = self.pairs[:subset_size]

        print(f"Dataset initialized with {len(self.pairs)} matched image/mask pairs")
        if unmatched:
            print(f"WARNING: {len(unmatched)} images had no matching mask. First few: {unmatched[:5]}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.pairs[idx]

        img = cv2.imread(os.path.join(self.image_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1)

        mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        mask = torch.from_numpy(mask.astype(np.int64))

        return img, mask

dataset = FloodDataset(image_dir, mask_dir, subset_size=SUBSET_SIZE, img_size=IMG_SIZE)

assert len(dataset) > 0, "Dataset has 0 pairs — re-check Section 3's overlap table, the selected pair may still be wrong."

sample_img, sample_mask = dataset[0]
print("Image tensor shape:", sample_img.shape, "dtype:", sample_img.dtype)
print("Mask tensor shape:", sample_mask.shape, "dtype:", sample_mask.dtype)
print("Mask unique values (these are the class IDs):", torch.unique(sample_mask))


**Check the mask unique values above.** `[0, 1]` means already binary. More values
means multiple classes — confirm which integer means "water" before collapsing to
binary with the cell below.


In [ ]:
# Only needed if you saw more than 2 unique values above.
WATER_CLASS_ID = 1  # confirm against FloodNet's class legend before trusting this

def to_binary_mask(mask, water_id=WATER_CLASS_ID):
    return (mask == water_id).long()

binary_sample = to_binary_mask(sample_mask)
print("Binary mask unique values:", torch.unique(binary_sample))
print("Fraction of pixels labeled water:", binary_sample.float().mean().item())


In [ ]:
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

imgs, masks = next(iter(loader))
print("Batch image shape:", imgs.shape)
print("Batch mask shape:", masks.shape)


## 7. Pretrained U-Net — zero-training sanity check
Output will look like noise — expected. Confirms the pipeline works end-to-end
before any training happens.


In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)
model = model.to(device)
model.eval()

with torch.no_grad():
    imgs_gpu = imgs.to(device)
    raw_output = model(imgs_gpu)
    pred_probs = torch.sigmoid(raw_output)

print("Model output shape:", raw_output.shape)
print("Output value range:", pred_probs.min().item(), "to", pred_probs.max().item())

del imgs_gpu, raw_output
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i in range(4):
    img_np = imgs[i].permute(1, 2, 0).numpy()
    mask_np = masks[i].numpy()
    pred_np = pred_probs[i, 0].cpu().numpy()

    axes[0, i].imshow(img_np)
    axes[0, i].set_title(f"Input {i}")
    axes[1, i].imshow(mask_np, cmap='Blues')
    axes[1, i].set_title(f"Ground truth {i}")
    axes[2, i].imshow(pred_np, cmap='Blues')
    axes[2, i].set_title(f"Untrained prediction {i}")
    for row in range(3):
        axes[row, i].axis('off')
plt.tight_layout()
plt.show()

print("Untrained predictions should look like noise — expected. Training happens in Phase 2.")


## 8. IoU and confusion matrix
Same concept as precision/recall from NIDS, applied per-pixel.


In [ ]:
def compute_iou(pred_mask, true_mask, threshold=0.5):
    pred_binary = (pred_mask > threshold).astype(np.uint8)
    true_binary = true_mask.astype(np.uint8)

    intersection = np.logical_and(pred_binary, true_binary).sum()
    union = np.logical_or(pred_binary, true_binary).sum()

    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return intersection / union

ious = []
for i in range(imgs.shape[0]):
    pred_np = pred_probs[i, 0].cpu().numpy()
    true_np = masks[i].numpy()
    iou = compute_iou(pred_np, true_np)
    ious.append(iou)
    print(f"Sample {i}: IoU = {iou:.4f}")

print(f"\nMean IoU (untrained baseline): {np.mean(ious):.4f}")
print("Expect this to be low/near-random right now — Phase 2 training improves it.")


In [ ]:
from sklearn.metrics import confusion_matrix

pred_flat = (pred_probs[0, 0].cpu().numpy() > 0.5).astype(np.uint8).flatten()
true_flat = masks[0].numpy().astype(np.uint8).flatten()

cm = confusion_matrix(true_flat, pred_flat, labels=[0, 1])
print("Confusion matrix (rows=actual, cols=predicted):")
print("              Pred: not-water   Pred: water")
print(f"Actual not-water   {cm[0,0]:>10}      {cm[0,1]:>10}")
print(f"Actual water       {cm[1,0]:>10}      {cm[1,1]:>10}")


## Phase 1 complete
At this point you have:
- FloodNet downloading with correctly-paired image/mask folders (chosen by ID overlap, not folder name guessing)
- A working PyTorch Dataset/DataLoader
- A pretrained U-Net running inference (untrained, expected to be noisy)
- Working IoU and confusion matrix code

**Next (Phase 2):** train the model and get a real baseline IoU score.
